<a href="https://colab.research.google.com/github/sensiwise-ai/agentic-ai-workshop2026-labs/blob/main/Lab3_Research_Agent_Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 Research Agent: from your question to a cited briefing

Lab 1 searched documents once. Lab 2 ran a fixed pipeline. Here the model **decides what to search for, which papers to read, whether it has learned enough, and when to stop**. That is the pattern behind every "deep research" product, and you will build it in six small functions over **arXiv itself**: 2.5 million papers, searched live through its API, any topic, any field, up to yesterday.

```
your question ─▶ 1 plan searches (model) ─▶ 2 find candidate papers (arXiv API) ─▶ 3 select the ones worth reading (model)
              ─▶ 4 download each one, read it, summarise how it works (model) ─▶ 5 assess coverage: enough? (model) ──no──▶ back to 2
                                                                                                           └──yes──▶ 6 write the briefing, verify citations
```

Finding is cheap: a search returns titles and abstracts in a second. Reading is expensive: a PDF must be downloaded and extracted. So the agent finds widely, reads narrowly, and only ever reads what it chose.

### Setup: Runtime → Change runtime type → **T4 GPU**, then run this cell

In [ ]:
!pip install -q -U transformers accelerate chromadb sentence-transformers pypdf arxiv

import torch, json, re
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"          # 4B open model, ~9 GB in fp16, fits a free T4. (Fallback: "Qwen/Qwen3-4B")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

In [ ]:
def chat(messages, max_new_tokens=400, temperature=0.0):
    """The one function that talks to the model: a prompt string or a list of {role, content} -> reply text."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                         temperature=temperature if temperature > 0 else None)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
print(chat("Say hello in five words."))

## 1: The source: arXiv, live

The `arxiv` package wraps arXiv's public API and respects its rate limit (one request every three seconds). A search returns metadata and the abstract for each paper; the PDF is only fetched later, for papers the agent selects. Everything the agent has seen in this session is kept in one dictionary, `PAPERS`, keyed by arXiv id.

In [ ]:
import arxiv, requests, io, re, pandas as pd
from pypdf import PdfReader
from IPython.display import Markdown, display

api = arxiv.Client()                                                                   # polite by default: 3 s between requests, retries
PAPERS = {}                                                                            # id -> {title, year, authors, abstract, url, pdf, text}

def step(n, title, *lines):
    '''One banner per pipeline step, so every run reads the same way.'''
    print(f"\n{'━' * 72}\n Step {n} · {title}\n{'━' * 72}")
    for line in lines:
        print(line)

def search_arxiv(query, k=5):
    '''Ask arXiv for the k papers most relevant to a query. Returns their ids; metadata + abstract go into PAPERS.'''
    ids = []
    for r in api.results(arxiv.Search(query=query, max_results=k, sort_by=arxiv.SortCriterion.Relevance)):
        pid = r.get_short_id().split("v")[0]
        PAPERS.setdefault(pid, {"id": pid, "title": r.title.replace("\n", " "), "year": r.published.year, "authors": ", ".join(a.name for a in r.authors),
                                "abstract": r.summary.replace("\n", " "), "url": f"https://arxiv.org/abs/{pid}", "pdf": r.pdf_url, "text": None})
        ids.append(pid)
    return ids

In [ ]:
search = search_arxiv                                                                  # the pipeline's source; swapped later for your own documents

for pid in search("mixture of experts inference cost", k=5):
    print(f"  {pid}  {PAPERS[pid]['year']}  {PAPERS[pid]['title'][:80]}")

In [ ]:
print(PAPERS)

## 2: Reading a paper: download, extract, index

`fetch_text()` downloads the PDF once and extracts its text with `pypdf`; `read_index()` chunks that text and embeds it into a **reading index** for just the selected papers; `passages()` pulls the few chunks of one paper closest to the topic. Extraction from PDFs is imperfect (two columns, equations, footers) — this is real-world text now, not a curated dataset.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedder = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda")
client = chromadb.Client()

In [ ]:
def fetch_text(pid):
    '''Download the PDF (once) and extract its text.'''
    paper = PAPERS[pid]
    if paper["text"] is None:
        pdf = requests.get(paper["pdf"], timeout=90).content
        text = "\n".join(page.extract_text() or "" for page in PdfReader(io.BytesIO(pdf)).pages)
        paper["text"] = re.sub(r"[ \t]+", " ", re.sub(r"-\n(?=[a-z])", "", text))     # re-join hyphenated words, collapse spaces
    return paper["text"]

def chunk(text, size=800):
    pieces, current = [], ""
    for word in text.split():
        if len(current) + len(word) > size:
            pieces.append(current.strip()); current = ""
        current += word + " "
    return pieces + [current.strip()]

def fresh_collection(name):
    '''A new, empty collection with that name (replacing any previous one).'''
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name, embedding_function=embedder, metadata={"hnsw:space": "cosine"})

def read_index(ids):
    '''The reading index: chunk and embed the full text of just these papers.'''
    reading = fresh_collection("reading")
    for pid in ids:
        pieces = chunk(fetch_text(pid))
        reading.add(ids=[f"{pid}-{i}" for i in range(len(pieces))], documents=pieces, metadatas=[{"id": pid, "title": PAPERS[pid]["title"]}] * len(pieces))
    return reading

def passages(reading, topic, pid, k=6):
    '''The k passages of one paper most relevant to the topic.'''
    return reading.query(query_texts=[topic], n_results=k, where={"id": pid})["documents"][0]

In [ ]:
pid = search("mixture of experts inference cost", k=1)[0]
reading = read_index([pid])
print(f"{PAPERS[pid]['title'][:70]}: {len(PAPERS[pid]['text']):,} characters, {reading.count()} chunks")
print(passages(reading, "how are experts selected at inference time", pid, k=1)[0][:10000], "...")

<details><summary><b>Fallback if arXiv is slow or down today</b> — set <code>USE_DATASET = True</code> and run</summary>

The <code>jamescalam/ai-arxiv2</code> dataset holds 2,673 AI papers (2016 to early 2024) with clean full text. This cell loads them into <code>PAPERS</code>, indexes their abstracts locally, and points <code>search</code> at that index. Everything after this cell works unchanged.
</details>

In [ ]:
USE_DATASET = False                                                                    # <- True only if arXiv is unavailable

def build_library(ids):
    '''A local search index over title + abstract of these papers (used instead of the arXiv API).'''
    library = fresh_collection("library")
    for i in range(0, len(ids), 500):
        batch = ids[i:i + 500]
        library.add(ids=batch, documents=[PAPERS[p]["title"] + ". " + PAPERS[p]["abstract"] for p in batch], metadatas=[{"title": PAPERS[p]["title"]} for p in batch])
    print(library.count(), "documents in the local library index")
    return library

def search_library(query, k=8):
    return library.query(query_texts=[query], n_results=min(k, library.count()))["ids"][0]

if USE_DATASET:
    from datasets import load_dataset
    for r in load_dataset("jamescalam/ai-arxiv2", split="train"):
        text = r["content"].encode("latin-1", "ignore").decode("utf-8", "ignore")
        PAPERS[r["id"]] = {"id": r["id"], "title": r["title"], "year": int(r["published"][:4]), "authors": r["authors"], "abstract": r["summary"].replace("\n", " "),
                           "url": f"https://arxiv.org/abs/{r['id']}", "pdf": None, "text": text[text.find("\n# "):] if "\n# " in text[:3000] else text}
    library = build_library(list(PAPERS))
    search = search_library

## 3: What do you want to know?

Type a question or a topic, in any field arXiv covers (physics, biology, economics, computer science...). Press Enter for the example.

In [ ]:
TOPIC = "Small Language models against large language models"

## Step 1: Plan the searches — the model's first decision

A topic is not a search query. The model breaks it into three or four different searches that together cover it. This is where the agent decides *what to look for*.

In [ ]:
def chat_json(messages, **kw):
    """Same as chat(), but returns the first JSON object found in the reply as a dict."""
    text = chat(messages, **kw)
    match = re.search(r"\{.*\}", text, re.S)
    return json.loads(match.group()) if match else {}

In [ ]:
def plan(topic):
    reply = chat_json(f"You research a topic in the arXiv paper archive, which matches keywords, not sentences. Write 3-4 different "
                      f"keyword searches of 2-5 technical terms each (no question words, no 'large language models' on its own) that "
                      f'together cover the topic. Return JSON: {{"queries": [...]}} where every query is a plain string.\n\nTopic: {topic}')
    return [" ".join(map(str, q.values())) if isinstance(q, dict) else str(q) for q in reply.get("queries", [topic])]   # small models sometimes return objects

queries = plan(TOPIC)
step(1, "Plan searches (model)", *queries)

## Step 2: Find candidate papers

Each query goes to the source and the hits are pooled. arXiv matches keywords, so its own ranking is weak for a topic; we **re-rank the pool by meaning**, embedding each title + abstract and scoring its similarity to the topic (Lab 1's trick). The ten best go forward. Cheap, fast, and deliberately generous: better to show the model ten candidates than to miss the right one.

In [ ]:
def rank_by_meaning(topic, ids, keep=10):
    '''Embed each candidate's title + abstract and rank them by similarity to the topic -> {paper id: similarity}, best first.'''
    pool = fresh_collection("candidates")
    pool.add(ids=ids, documents=[PAPERS[p]["title"] + ". " + PAPERS[p]["abstract"] for p in ids])
    r = pool.query(query_texts=[topic], n_results=min(keep, len(ids)))
    return {pid: round(1 - d, 2) for pid, d in zip(r["ids"][0], r["distances"][0])}

def find_candidates(topic, queries, k=6, exclude=()):
    '''Pool the papers found by every query (de-duplicated), then re-rank the pool by meaning.'''
    pool = list(dict.fromkeys(pid for q in queries for pid in search(q, k) if pid not in exclude))
    return rank_by_meaning(topic, pool) if pool else {}

candidates = find_candidates(TOPIC, queries)
step(2, "Find candidates (re-ranked by meaning)", *[f"{score:.2f}  {pid}  {PAPERS[pid]['year']}  {PAPERS[pid]['title'][:75]}" for pid, score in candidates.items()])

## Step 3: Select the papers worth reading — the model's second decision

Similarity ranks by wording; the model ranks by usefulness. It reads the candidates' abstracts and picks the few that will actually answer the topic, with a reason for each. Reading is the expensive step, so this choice matters.

In [ ]:
def select_papers(topic, candidates, n=4):
    '''The model picks the n most useful candidates -> [{"id", "why"}, ...]. Unknown ids are dropped.'''
    listing = "\n".join(f"{pid}: {PAPERS[pid]['title']} ({PAPERS[pid]['year']}) — {PAPERS[pid]['abstract'][:300]}" for pid in candidates)
    reply = chat_json(f"Topic: {topic}\n\nCandidate papers:\n{listing}\n\nPick the {n} papers most useful for a briefing on the topic, "
                      f'most useful first. Return JSON: {{"papers": [{{"id": "...", "why": "..."}}]}}', max_new_tokens=400)
    chosen = [p for p in reply.get("papers", []) if p.get("id") in candidates][:n]
    return chosen or [{"id": pid, "why": "highest similarity to the topic"} for pid in list(candidates)[:n]]

chosen = select_papers(TOPIC, candidates)
step(3, "Select papers (model)", *[f"{PAPERS[c['id']]['title'][:80]}\n     why: {c['why']}" for c in chosen])

### Something to highlight here

One of the papers the model picked is about the **tackling the critical environmental and economic**, which may not be the most relevant choice.

This happened because the query never specified which aspects we care about most.

> **Rule of thumb:** the vaguer your query → the more generic the answers you get.

## Step 4: Read each paper and summarise how it works — the model's third decision

For every selected paper: download and extract the PDF, build the reading index, pull the six passages most relevant to the topic, and write a structured summary from the abstract and those passages only. Problem, mechanism, results, relevance, limitations. One summary per paper, so nothing gets blended together yet.

In [ ]:
def summarize_paper(topic, pid, passages):
    paper = PAPERS[pid]
    joined = "\n\n".join(f"- {p[:900]}" for p in passages)
    return chat(f"You are summarising one research paper for a briefing on: {topic}\n\nPaper: {paper['title']} ({paper['year']})\n"
                f"Abstract: {paper['abstract'][:1200]}\n\nRelevant passages:\n{joined}\n\n"
                f"Using only this material, write under these bold headings: **Problem**, **How it works** (3-4 sentences on the mechanism), "
                f"**Key results** (numbers if given), **Relevance to the topic**, **Limitations**. Maximum 200 words.", max_new_tokens=380)

summaries = {}                                                                          # title -> summary text
reading = read_index([c["id"] for c in chosen])                                         # downloads the PDFs: ~5-10 s each
for c in chosen:
    title = PAPERS[c["id"]]["title"]
    summaries[title] = summarize_paper(TOPIC, c["id"], passages(reading, TOPIC, c["id"]))
    step(4, f"Read & summarise (model) : {title[:60]}"); display(Markdown(summaries[title]))

## Step 5: Assess coverage: enough, or search again? — the model's fourth decision

This is what a plain pipeline cannot do. The model reads its own summaries, names what the topic still lacks, and proposes new searches for those gaps, or says it has enough. Forcing named gaps and concrete queries stops it from just saying "yes, done".

In [ ]:
def assess(topic, summaries, used_queries):
    joined = "\n\n".join(f"### {title}\n{text[:700]}" for title, text in summaries.items())
    return chat_json(f"Topic: {topic}\n\nPaper summaries gathered so far:\n{joined}\n\nSearch queries already used: {used_queries}\n\n"
                     f"Which aspects of the topic are still not covered by these papers? Propose at most 2 NEW keyword searches for the gaps "
                     f"(plain strings), or none if the summaries already cover the topic well.\n"
                     f'Return JSON: {{"gaps": [...], "new_queries": [...], "enough": true/false}}', max_new_tokens=300)

audit = assess(TOPIC, summaries, queries)
step(5, "Assess coverage (model)", f"gaps: {audit.get('gaps') or 'none'}", f"enough: {audit.get('enough')}", f"new queries: {audit.get('new_queries') or 'none'}")

## Step 6: Write the briefing and verify the citations

The briefing is written from the summaries, not from raw chunks, so the prompt stays small and each paper is represented fairly. Code checks which of the papers actually read the briefing cites, flags anything cited that was never read, and appends the reference list with real arXiv links.

In [ ]:
def write_briefing(topic, summaries):
    joined = "\n\n".join(f"### {title}\n{text}" for title, text in summaries.items())
    return chat(f"Write a literature briefing on: {topic}\n\nUse ONLY the paper summaries below. Structure:\n"
                f"1. **Overview** (3 sentences)\n2. **How the approaches work**: one short paragraph per paper, citing its exact title in square brackets\n"
                f"3. **Comparison**: a markdown table with columns Paper | Approach | Trade-off\n4. **What is still open** (2-3 bullets)\n"
                f"Maximum 450 words.\n\n{joined}", max_new_tokens=900)

def verify(briefing, summaries, refs):
    '''Which of the papers actually read does the briefing cite, and does it cite anything else?'''
    cited = [title for title in summaries if title.lower() in briefing.lower()]
    unknown = [c for c in re.findall(r"\[([^\]]+)\]", briefing) if c not in summaries]
    references = "\n".join(f"- {title} — {refs[title]}" for title in summaries)
    return f"cites {len(cited)} of the {len(summaries)} papers read, {len(unknown)} unknown citation{'s' if len(unknown) != 1 else ''}{': ' + str(unknown) if unknown else ''}", briefing + "\n\n**References**\n" + references

refs = {PAPERS[c["id"]]["title"]: PAPERS[c["id"]]["url"] for c in chosen}
check, final = verify(write_briefing(TOPIC, summaries), summaries, refs)
step(6, "Briefing (model) + citation check", check); display(Markdown(final))

## The whole agent

Steps 1 to 6 in one function, with the loop that makes it an agent: after step 5, if the model names gaps and new queries, the agent goes back to step 2 for those gaps, reads more papers, and assesses again, until it says *enough* or the round budget runs out. Budgets always beat the model's judgement.

In [ ]:
def research(topic, max_rounds=2, per_round=4):
    print(f"\nTOPIC: {topic}")
    queries, used, read_ids, summaries, refs = plan(topic), [], set(), {}, {}
    step(1, "Plan searches (model)", *queries)
    for round_no in range(1, max_rounds + 1):
        candidates = find_candidates(topic, queries, exclude=read_ids)                 # 2 · never re-read a paper
        used += queries;   step(2, f"Find candidates · round {round_no}", *[f"{score:.2f}  {PAPERS[pid]['year']}  {PAPERS[pid]['title'][:75]}" for pid, score in candidates.items()])
        if not candidates:
            step(6, "Briefing", "No papers found for these searches - rephrase the topic."); return summaries
        chosen = select_papers(topic, candidates, n=per_round if round_no == 1 else 2)  # 3
        step(3, "Select papers (model)", *[f"{PAPERS[c['id']]['title'][:80]}\n     why: {c['why']}" for c in chosen])
        reading = read_index([c["id"] for c in chosen])                                # 4 · download + extract + index
        for c in chosen:
            title = PAPERS[c["id"]]["title"]
            summaries[title] = summarize_paper(topic, c["id"], passages(reading, topic, c["id"]))
            refs[title] = PAPERS[c["id"]]["url"]; read_ids.add(c["id"])
            step(4, f"Read & summarise (model) · {title[:60]}"); display(Markdown(summaries[title]))
        audit = assess(topic, summaries, used)                                         # 5
        step(5, "Assess coverage (model)", f"gaps: {audit.get('gaps') or 'none'}", f"enough: {audit.get('enough')}", f"new queries: {audit.get('new_queries') or 'none'}")
        if audit.get("enough") or not audit.get("new_queries"):
            break
        queries = [str(q) for q in audit["new_queries"]]
    check, final = verify(write_briefing(topic, summaries), summaries, refs)            # 6
    step(6, "Briefing (model) + citation check", f"{len(summaries)} papers read · {check}"); display(Markdown(final))
    return summaries

summaries = research(TOPIC)

**What made it agentic.** The model chose the searches, chose the papers, judged its own coverage and decided whether to go round again; the code only fixed the order of steps, the budgets (`max_rounds`, `per_round`) and the checks (real ids only, real citations only, references from code). Compare with Lab 2, where the model never decided *whether* to take another step.

## The same agent as a graph — LangGraph

`research()` is a loop with a budget and one decision inside it. Written as a graph, the loop becomes a **cycle** with a conditional edge, the budget becomes a routing rule, and the state after every step is something you can inspect, checkpoint or resume. The six functions do not change; the graph only decides which one runs next.

In [ ]:
!pip install -q "langgraph>=0.6"

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

MAX_ROUNDS = 1                                                             # one round for this demo; research() used two

class Research(TypedDict, total=False):                                    # everything research() kept in local variables
    topic: str
    queries: list
    used: list
    read_ids: list
    summaries: dict
    refs: dict
    round: int
    audit: dict
    briefing: str

def plan_node(s):
    queries = plan(s["topic"]); step(1, "Plan searches (model)", *queries)
    return {"queries": queries, "used": [], "read_ids": [], "summaries": {}, "refs": {}, "round": 0}

def read_node(s):                                                          # steps 2, 3 and 4: one round of finding and reading
    round_no = s["round"] + 1
    candidates = find_candidates(s["topic"], s["queries"], exclude=set(s["read_ids"]))
    step(2, f"Find candidates · round {round_no}", *[f"{score:.2f}  {PAPERS[pid]['year']}  {PAPERS[pid]['title'][:75]}" for pid, score in candidates.items()])
    summaries, refs, read_ids = dict(s["summaries"]), dict(s["refs"]), list(s["read_ids"])
    if candidates:
        chosen = select_papers(s["topic"], candidates, n=4 if round_no == 1 else 2)
        step(3, "Select papers (model)", *[f"{PAPERS[c['id']]['title'][:80]}\n     why: {c['why']}" for c in chosen])
        reading = read_index([c["id"] for c in chosen])
        for c in chosen:
            title = PAPERS[c["id"]]["title"]
            summaries[title] = summarize_paper(s["topic"], c["id"], passages(reading, s["topic"], c["id"]))
            refs[title] = PAPERS[c["id"]]["url"]; read_ids.append(c["id"])
            step(4, f"Read & summarise (model) · {title[:60]}"); display(Markdown(summaries[title]))
    return {"round": round_no, "summaries": summaries, "refs": refs, "read_ids": read_ids, "used": s["used"] + s["queries"]}

def assess_node(s):
    audit = assess(s["topic"], s["summaries"], s["used"]) if s["summaries"] else {"enough": True, "gaps": ["no papers found"]}
    step(5, "Assess coverage (model)", f"gaps: {audit.get('gaps') or 'none'}", f"enough: {audit.get('enough')}", f"new queries: {audit.get('new_queries') or 'none'}")
    return {"audit": audit, "queries": [str(q) for q in audit.get("new_queries") or []]}

def brief_node(s):
    check, final = verify(write_briefing(s["topic"], s["summaries"]), s["summaries"], s["refs"])
    step(6, "Briefing (model) + citation check", f"{len(s['summaries'])} papers read · {check}"); display(Markdown(final))
    return {"briefing": final}

def enough(s):                                                             # the one decision the loop was made of
    return "brief" if s["audit"].get("enough") or not s["queries"] or s["round"] >= MAX_ROUNDS else "read"

graph = StateGraph(Research)
for name, fn in [("plan", plan_node), ("read", read_node), ("assess", assess_node), ("brief", brief_node)]:
    graph.add_node(name, fn)
graph.add_edge(START, "plan")
graph.add_edge("plan", "read")
graph.add_edge("read", "assess")
graph.add_conditional_edges("assess", enough, {"read": "read", "brief": "brief"})
graph.add_edge("brief", END)
researcher = graph.compile()

In [ ]:
from IPython.display import Image
try:
    display(Image(researcher.get_graph().draw_mermaid_png()))               # the loop, drawn by the library from the edges
except Exception:
    print(researcher.get_graph().draw_mermaid())                            # the same diagram as text, if the renderer is unreachable

In [ ]:
result = researcher.invoke({"topic": TOPIC})                              # same functions, same banners; papers already fetched are reused

Same six functions, same banners, same budget. What changed is that the loop is now a drawing, the state after every node can be saved and resumed, and a second person can read the control flow without reading `research()`. When the shape of an agent is a cycle with a decision in it, this is the tool built for that shape.

## 🗂️ Your turn: your own papers or reports

Upload the documents you actually work with (`.pdf .txt .md`): papers, reports, proposals, standards. They go into `PAPERS` (the first 1,200 characters stand in for the abstract), a local index over them replaces arXiv as the source, and the same `research()` runs with the same banners. Nothing leaves this Colab session.

In [ ]:
from google.colab import files
from pypdf import PdfReader

def upload():
    """Open the file picker; return {filename: text} for .pdf / .txt / .md / .csv files."""
    texts = {}
    for name, blob in files.upload().items():
        if name.lower().endswith(".pdf"):
            open(name, "wb").write(blob)
            texts[name] = "\n".join(page.extract_text() or "" for page in PdfReader(name).pages)
        else:
            texts[name] = blob.decode("utf-8", errors="ignore")
    print({k: f"{len(v)} chars" for k, v in texts.items()})
    return texts

In [ ]:
mine = upload()
for name, text in mine.items():
    PAPERS[name] = {"id": name, "title": name, "year": 2026, "authors": "", "abstract": text[:1200].replace("\n", " "), "url": name, "pdf": None, "text": text}
library = build_library(list(mine))                                                    # a local index over YOUR documents ...
search = search_library                                                                # ... and it becomes the source. research() is unchanged.
summaries = research(input("What do you want to know about these documents? ").strip() or "What problem do these documents address, how do their approaches work, and how do they differ?")

In [ ]:
search = search_arxiv                                                                  # back to arXiv; another topic, 'quit' to stop
while True:
    q = input("Research topic: ").strip()
    if q.lower() in ("quit", "exit", "q", ""): break
    research(q)

#### If the briefing is thin, read the banners in order: were the step 1 searches good keyword queries for your topic? Did step 2 find papers with similarity above about 0.5 (arXiv matches keywords: rephrase the searches)? Did step 3 pick the ones you would have picked? Was the extracted text in step 4 readable? Each step has its own fix.

## Exercises & Recap
1. **Budget**: run `research(TOPIC, max_rounds=1)` and `max_rounds=3`. What changed in the briefing, and what did it cost in minutes?
2. **Recency**: sort by `arxiv.SortCriterion.SubmittedDate` in `search_arxiv` and ask for recent work only. What does relevance lose?
3. **A critic**: after step 6, ask the model to list every claim in the briefing that is not supported by a summary. Where would you put that in `research()`?
4. **Threshold**: drop candidates below 0.4 similarity in step 2 and let the agent say "arXiv has little on this" instead of reading weak papers. Where does the message belong?

| Step | Who decides | Function |
|---|---|---|
| 1 Plan searches | **model** | `plan()` |
| 2 Find candidates, re-rank by meaning | code · arXiv API (or your local index) + embeddings | `find_candidates()`, `rank_by_meaning()` |
| 3 Select papers | **model** | `select_papers()` |
| 4 Download, read and summarise | **model**, reading index | `fetch_text()`, `read_index()`, `passages()`, `summarize_paper()` |
| 5 Assess coverage, loop or stop | **model**, bounded by `max_rounds` | `assess()` |
| 6 Briefing and citation check | **model**, then code | `write_briefing()`, `verify()` |

*Papers come from [arXiv](https://arxiv.org) through its public API; each keeps its own licence. Fallback dataset: [jamescalam/ai-arxiv2](https://huggingface.co/datasets/jamescalam/ai-arxiv2).*